# 3D Packing Problem

#### Importing packages

In [34]:
import numpy as np 
import pandas as pd
from amplpy import AMPL

In [35]:
pd.set_option('display.max_rows', 200)

### Notations

- $I=\{1,2,...,N\}$ is the set of SKUs, where N denotes the total number of SKUs to be packed.

- $J=\{1,2,...,M\}$ is the set of DFCs, M denotes the number of different types of DFCs available for packing.
  
- $T_{j}$ is the set of DFCs of type j, $\forall  j \in J$

- The $i^{th}$ SKU is characterized by its length $l_i$, width $w_i$, height $h_i$, weight $m_i$. $\forall i \in I$

- The $j^{th}$ DFC is characterized by its length $L_j$, width $W_j$, height $H_j$ and maximum weight carrying capacity $G_j$. $\forall j \in J$

- $lr^+_{ii′} = 1$, if $i^{th}$ SKU is placed on left-hand side of $i′^{th}$ SKU; otherwise, $lr^+_{ii′} = 0$.
- $bf^+_{ii′} = 1$, if $i^{th}$ SKU is placed behind the $i′^{th}$ SKU;    otherwise, $bf^+_{ii′} = 0$.
- $bt^+_{ii′} = 1$, if $i^{th}$ SKU is placed on below the $i′^{th}$ SKU; otherwise, $bt^+_{ii′} = 0$.
- $lr^-_{ii′} = 1$, if $i^{th}$ SKU is placed on right-hand side of $i′^{th}$  SKU; otherwise, $lr^-_{ii′} = 0$.
- $bf^-_{ii′} = 1$, if $i^{th}$ SKU is placed in front of $i′^{th}$ SKU; otherwise, $bf^-_{ii′} = 0$.
- $bt^-_{ii′} = 1$, if $i^{th}$ SKU is placed above the $i′^{th}$ SKU; otherwise, $bt^-_{ii′} = 0$.

- $u_{jk}$ is a binary variable, which denotes whether $k_{th}$ copy of the $j_{th}$ DFC is being used for packaging.

- $s_{ijk}$ is a binary variable to denote, whether the $i_{th}$ SKU is packed in $k_{th}$ copy of
the $j_{th}$ DFC.

- The variables $x_i$, $y_i$, $z_i$ denotes the coodinated of the left-bottom-back corner of
$i_{th}$ SKU.

- The Variables $l^x_i$ , $l^y_i$ , $l^z_i$ denotes whether the length edge of ith SKU is parallel to X-axis, Y-axis or Z-axis.

- Similarly, the variables $w^x_i$ , $w^y_i$ , $w^z_i$ denotes whether the width edge of ith SKU is parallel to X-axis, Y-axis or Z-axis, and the variables $h^x_i$ , $h^y_i$ , $h^z_i$ denotes whether the height edge of $i_{th}$ SKU is parallel to X-axis, Y-axis or Z-axis.


### Formulation

The objective is to minimize the total vacant spaces or unused volume within
the assigned DFCs or, equivalently, the total volume of the chosen DFCs for
packing. The latter is easier to model. The overall model is as follows: 

$$
\begin{align}

\min \quad 
& \sum_{j\in J}\sum_{k\in T_j} u_{jk} L_j W_j H_j 
- \sum_{i\in I} l_i w_i h_i \\

\text{s.t.} \quad 
& \sum_{j\in J}\sum_{k\in T_j} s_{ijk} = 1 
&& \forall i \in I \\

& u_{jk} \ge s_{ijk}
&& \forall i \in I,\; \forall j \in J,\; \forall k \in T_j \\

& l_i^x + l_i^y + l_i^z = 1
&& \forall i \in I \\

& w_i^x + w_i^y + w_i^z = 1
&& \forall i \in I \\

& h_i^x + h_i^y + h_i^z = 1
&& \forall i \in I \\

& l_i^x + w_i^x + h_i^x = 1
&& \forall i \in I \\

& l_i^y + w_i^y + h_i^y = 1
&& \forall i \in I \\

& l_i^z + w_i^z + h_i^z = 1
&& \forall i \in I \\

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\le L_j + (1-s_{ijk})M
&& \forall i\in I,\forall j\in J,\forall k\in T_j \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\le W_j + (1-s_{ijk})M
&& \forall i\in I,\forall j\in J,\forall k\in T_j \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\le H_j + (1-s_{ijk})M
&& \forall i\in I,\forall j\in J,\forall k\in T_j \\

& \sum_{i\in I} s_{ijk} m_i \le G_j
&& \forall j\in J,\forall k\in T_j \\

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\le x_{i'} + (1-lr_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& x_{i'} + l_{i'}^x l_{i'} + w_{i'}^x w_{i'} + h_{i'}^x h_{i'}
\le x_i + (1-lr_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\le y_{i'} + (1-bf_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& y_{i'} + l_{i'}^y l_{i'} + w_{i'}^y w_{i'} + h_{i'}^y h_{i'}
\le y_i + (1-bf_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\le z_{i'} + (1-bt_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& z_{i'} + l_{i'}^z l_{i'} + w_{i'}^z w_{i'} + h_{i'}^z h_{i'}
\le z_i + (1-bt_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& lr_{ii'}^{+} + lr_{ii'}^{-}
+ bf_{ii'}^{+} + bf_{ii'}^{-}
+ bt_{ii'}^{+} + bt_{ii'}^{-}
\ge s_{ijk} + s_{i'jk} - 1
&& i<i',\forall i,i'\in I,\forall j\in J,\forall k\in T_j

\end{align}
$$


#### Adding constraint so that the lowest indexed copy of dfc is used first(but not used here)
$$
\begin{align}
u_{jp} \ge u_{jq}
\qquad
\forall j \in J,\;
p,q \in T_j: q = p + 1,\;
\end{align}
$$

- Just add the code below in model function before solve to apply the constraint:


    ampl.eval(
        """
        s.t. c20 {j in DFC, p in Copy[j], q in Copy[j] : q = p+1}:
        copy_used[j,p] >= copy_used[j,q];
        """
        )

fix command to fix a solution 

In [36]:
def model(a):
    with pd.ExcelFile(a) as xls:
        dfc_df = pd.read_excel(xls,"DFC")
        sku_df = pd.read_excel(xls, "SKU")

    print('dfc', dfc_df.to_string(), sep='\n')
    print('sku', sku_df.to_string(), sep='\n')


    # Initialize ampl object
    ampl = AMPL()

    # Make model
    ampl.eval(
        """
        set SKU;             
        set DFC;              
        set Copy{DFC};                            # copies of j_th DFC 
        set axis= {'x', 'y', 'z'};
        set dim= {'Length', 'Width', 'Height'};

        param dim_sku {SKU, dim};
        param sku_Weight {SKU};
        param dim_dfc {DFC, dim};
        param dfc_Weight {DFC};
        param M;       

        var relative_position_left {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_right {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_back {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_front {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_below {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_above {{i in SKU, l in SKU : i<l}} binary;

        var copy_used {j in DFC, k in Copy[j]} binary;
        var sku_in_copy {i in SKU, j in DFC, k in Copy[j]} binary;
        var sku_position {SKU, axis} >=0;

        var Length_orientation {SKU, axis} binary;
        var Width_orientation {SKU, axis} binary;
        var Height_orientation {SKU, axis} binary;

        minimize vacant_space:
        sum{j in DFC} sum{k in Copy[j]} copy_used[j,k] *dim_dfc[j,'Length']*dim_dfc[j,'Width']*dim_dfc[j,'Height'] - sum{i in SKU} dim_sku[i,'Length']*dim_sku[i,'Width']*dim_sku[i,'Height']; 

        s.t. c1 {i in SKU}: 
        sum{j in DFC} sum{k in Copy[j]} sku_in_copy[i,j,k] = 1;

        s.t. c2 {i in SKU}:
        Length_orientation[i,'x'] + Length_orientation[i,'y'] + Length_orientation[i,'z'] = 1;

        s.t. c3 {i in SKU}:
        Width_orientation[i,'x'] + Width_orientation[i,'y'] + Width_orientation[i,'z'] = 1;

        s.t. c4 {i in SKU}:
        Height_orientation[i,'x'] + Height_orientation[i,'y'] + Height_orientation[i,'z'] = 1;

        s.t. c5 {i in SKU}:
        Length_orientation[i,'x'] + Width_orientation[i,'x'] + Height_orientation[i,'x'] = 1;

        s.t. c6 {i in SKU}:
        Length_orientation[i,'y'] + Width_orientation[i,'y'] + Height_orientation[i,'y'] = 1;

        s.t. c7 {i in SKU}:
        Length_orientation[i,'z'] + Width_orientation[i,'z'] + Height_orientation[i,'z'] = 1;

        s.t. c8 {i in SKU, j in DFC, k in Copy[j]}:  
        sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= dim_dfc[j, 'Length'] + (1- sku_in_copy[i,j,k])*M;
        
        s.t. c9 {i in SKU, j in DFC, k in Copy[j]}: 
        sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= dim_dfc[j, 'Width'] + (1- sku_in_copy[i,j,k])*M;
        
        s.t. c10 {i in SKU, j in DFC, k in Copy[j]}: 
        sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= dim_dfc[j, 'Height'] + (1- sku_in_copy[i,j,k])*M;
        
        s.t. c11 {i in SKU, j in DFC, k in Copy[j]}: 
        copy_used[j,k] >= sku_in_copy[i,j,k];

        s.t. c12 {j in DFC, k in Copy[j]}:
        sum{i in SKU} sku_in_copy[i,j,k]*sku_Weight[i] <= dfc_Weight[j];

        s.t. c13 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= sku_position[l,'x'] + (1- relative_position_left[i,l])*M;
        
        s.t. c14 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'x'] + Length_orientation[l,'x']*dim_sku[l,'Length'] + Width_orientation[l,'x']*dim_sku[l,'Width'] + Height_orientation[l,'x']*dim_sku[l,'Height'] <= sku_position[i,'x'] + (1- relative_position_right[i,l])*M;

        s.t. c15 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= sku_position[l,'y'] + (1- relative_position_back[i,l])*M;
        
        s.t. c16 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'y'] + Length_orientation[l,'y']*dim_sku[l,'Length'] + Width_orientation[l,'y']*dim_sku[l,'Width'] + Height_orientation[l,'y']*dim_sku[l,'Height'] <= sku_position[i,'y'] + (1- relative_position_front[i,l])*M;

        s.t. c17 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= sku_position[l,'z'] + (1- relative_position_below[i,l])*M;
        
        s.t. c18 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'z'] + Length_orientation[l,'z']*dim_sku[l,'Length'] + Width_orientation[l,'z']*dim_sku[l,'Width'] + Height_orientation[l,'z']*dim_sku[l,'Height'] <= sku_position[i,'z'] + (1- relative_position_above[i,l])*M;
        
        s.t. c19 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l}:
        relative_position_left[i,l] + relative_position_right[i,l] + relative_position_back[i,l] + relative_position_front[i,l] + relative_position_below[i,l] + relative_position_above[i,l] >= sku_in_copy[i,j,k] + sku_in_copy[l,j,k] -1;

        """
        )

    # Feeding data to the model
    ampl.set['SKU'] = sku_df['sku']
    ampl.set['DFC'] = dfc_df['Type']
    ampl.set['Copy'] = {
        dfc_df['Type'][i]: list(range(dfc_df['Number of Copies'][i]))
        for i in range(len(dfc_df))
    }

    ampl.param['dim_sku'] = sku_df.set_index('sku').loc[:,['Length', 'Width', 'Height']]
    ampl.param['sku_Weight'] = sku_df.set_index('sku')['Weight']
    ampl.param['dim_dfc'] = dfc_df.set_index('Type').loc[:,['Length', 'Width', 'Height']]
    ampl.param['dfc_Weight'] = dfc_df.set_index('Type')['Weight']
    ampl.param['M'] = np.max(sku_df.loc[:,['Length', 'Width', 'Height']])

    # solve
    ampl.option['solver'] = 'gurobi'
    ampl.option['timelimit'] = 5
    ampl.solve()

    # get results
    copy_used_df = ampl.get_variable('copy_used').to_pandas()
    sku_in_copy_df = ampl.get_variable('sku_in_copy').to_pandas()
    sku_position_df = ampl.get_variable('sku_position').to_pandas()
    Length_orientation_df = ampl.get_variable('Length_orientation').to_pandas()
    Width_orientation_df = ampl.get_variable('Width_orientation').to_pandas()
    Height_orientation_df = ampl.get_variable('Height_orientation').to_pandas()

    objective = ampl.get_value('vacant_space')
    print(f'Objective function value: {objective}')

    return copy_used_df, sku_in_copy_df, sku_position_df, Length_orientation_df, Width_orientation_df, Height_orientation_df

In [37]:
model('3dpack.xlsx')

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 2       4      4       4      16
1     2                 3       2      2       2       4
sku
   sku  Length  Width  Height  Weight
0    1       2      2       2       2
1    2       1      1       1       1
2    3       2      2       2       2
3    4       1      1       1       1
4    5       2      2       2       2
5    6       2      2       2       2
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 118
25 simplex iterations
1 branching node
Objective function value: 118


(               copy_used.val
 index0 index1               
 1      0                   1
        1                   1
 2      0                   1
        1                   1
        2                   1,
                       sku_in_copy.val
 index0 index1 index2                 
 1      1      0                     0
               1                     0
        2      0                     0
               1                     0
               2                     1
 2      1      0                     0
               1                     0
        2      0                     1
               1                     0
               2                     0
 3      1      0                     0
               1                     0
        2      0                     0
               1                     1
               2                     0
 4      1      0                     0
               1                     0
        2      0                     1
         

#### Verify the results

In [38]:
verify_df = pd.DataFrame({'Sku': [1,2,3,4,5,6], 'copy': ['2(2)', '2(0)', '2(1)', '2(0)', '1(1)', '1(0)'], 'vacant space':['8-8=0', '8-1=7', '8-8=0', '7-1=6', '64-8=56', '64-8=56']})
total_vacant_space = 6+56+56
print(verify_df.to_string())
print(f'total vacant space: {total_vacant_space}')

   Sku  copy vacant space
0    1  2(2)        8-8=0
1    2  2(0)        8-1=7
2    3  2(1)        8-8=0
3    4  2(0)        7-1=6
4    5  1(1)      64-8=56
5    6  1(0)      64-8=56
total vacant space: 118


#### Edge cases
1. one sku in one dfc
2. all sku in one dfc

In [39]:
model('3dpack1.xlsx')

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 4       3      3       3       4
1     2                 5       4      4       4       4
sku
   sku  Length  Width  Height  Weight
0    1       3      3       3       3
1    2       3      3       3       3
2    3       4      4       3       3
3    4       3      3       3       3
4    5       3      4       4       3
5    6       2      3       3       3
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 41
34 simplex iterations
1 branching node
Objective function value: 41


(               copy_used.val
 index0 index1               
 1      0                   1
        1                   1
        2                   1
        3                   1
 2      0                   1
        1                   0
        2                   0
        3                   0
        4                   1,
                       sku_in_copy.val
 index0 index1 index2                 
 1      1      0                     0
               1                     0
               2                     0
               3                     1
        2      0                     0
               1                     0
               2                     0
               3                     0
               4                     0
 2      1      0                     1
               1                     0
               2                     0
               3                     0
        2      0                     0
               1                     0
      

In [40]:
model('3dpack2.xlsx')

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 2       2      2       2       5
sku
   sku  Length  Width  Height  Weight
0    1       1      1       1       1
1    2       1      1       1       1
2    3       1      1       1       1
3    4       1      1       1       1
Gurobi 12.0.3:Gurobi 12.0.3: infeasible problem
0 simplex iterations

suffix dunbdd OUT;
Objective function value: -4


(               copy_used.val
 index0 index1               
 1      0                   0
        1                   0,
                       sku_in_copy.val
 index0 index1 index2                 
 1      1      0                     0
               1                     0
 2      1      0                     0
               1                     0
 3      1      0                     0
               1                     0
 4      1      0                     0
               1                     0,
                sku_position.val
 index0 index1                  
 1      x                      0
        y                      0
        z                      0
 2      x                      0
        y                      0
        z                      0
 3      x                      0
        y                      0
        z                      0
 4      x                      0
        y                      0
        z                      0,
                Length_or

#### there is some problem with the formulation which is why the output is infeasible but in reality it is feasible and will use 1 dfc to pack all the 4 skus.